# **MODELO TFT PREDITIVO**
*by Miguel Ferreira*

Antes de começarmos a criar nosso modelo, devemos seguir este pequeno passo-a-passo para garantir o **isolamento do ambiente de desenvolvimento através da criação de um ambiente virtual e kernel próprios do projeto**. Segue-se o procedimento.

## Setup de Ambiente Python Isolado (venv + Jupyter)

Este guia descreve, de forma direta e reprodutível, como configurar um ambiente Python isolado para execução de notebooks com kernel próprio.

---

### 1. Verificação inicial do Python

No terminal, verificar instalações disponíveis:

```powershell
where.exe python
python --version
py --version
```

---

### 2. Criação do ambiente virtual

A partir da raiz do projeto:

```powershell
cd C:\projects\Libellula
py -3.11 -m venv .venv
```

---

### 3. Ativação do ambiente

```powershell
.\.venv\Scripts\activate
```

Confirmação:

```
(.venv)
```

---

### 4. Validação do ambiente isolado

```powershell
python --version
where.exe python
```

Validação adicional:

```powershell
python -c "import sys; print(sys.executable)"
```

O caminho retornado deve apontar para o diretório `.venv`.

---

### 5. Instalação do kernel do Jupyter

Com o ambiente ativo:

```powershell
pip install ipykernel
python -m ipykernel install --user --name tft_env --display-name "Python (tft_env)"
```

---

### 6. Inicialização do Jupyter

```powershell
jupyter lab
```

---

### 7. Seleção do kernel

No notebook, selecionar:

```
Python (tft_env)
```

---

### 8. Verificação dentro do notebook

```python
import sys
print(sys.executable)
```

O caminho deve corresponder ao ambiente `.venv`.

---

### Resultado

Ambiente Python isolado, com kernel próprio, pronto para execução de notebooks e desenvolvimento reprodutível.

Cumpridas todas as etapas acima, seguimos com o teste final básico:

In [1]:
import sys
print(sys.executable)

C:\projects\Libellula\.venv\Scripts\python.exe


Tudo ok. Estamos prontos para começar.

## Preparativos

Começamos pela importação das bibliotecas, mas, antes, precisamos estabelecer como as instalações das bibliotecas e dependências deve ser feita de forma mais eficiente. **Atenção: tudo isto deve ser feito no Powershell (ou em qualquer outro terminal de sua preferência).**

1. Ativamos o ambiente virtual:
   
   ``` powershell
   .\.venv\Scripts\activate
   ```
2. Depois, fazemos as instalações:

   
   ``` powershell
   pip install pandas numpy torch pytorch-lightning pytorch-forecasting matplotlib pyarrow
   ```  
3. Por fim, testamos se e quais instalações foram feitas:

   
   ``` powershell
   pip list
   ```   
   Para este último comando, tivemos o seguinte output:
```
Package                 Version
----------------------- -----------
aiohappyeyeballs        2.6.1
aiohttp                 3.13.5
aiosignal               1.4.0
asttokens               3.0.1
attrs                   26.1.0
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.20
decorator               5.2.1
executing               2.2.1
filelock                3.29.0
fonttools               4.62.1
frozenlist              1.8.0
fsspec                  2026.4.0
idna                    3.13
ipykernel               7.2.0
ipython                 9.13.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
Jinja2                  3.1.6
joblib                  1.5.3
jupyter_client          8.8.0
jupyter_core            5.9.1
kiwisolver              1.5.0
lightning               2.6.1
lightning-utilities     0.15.3
MarkupSafe              3.0.3
matplotlib              3.10.9
matplotlib-inline       0.2.1
mpmath                  1.3.0
multidict               6.7.1
nest-asyncio            1.6.0
networkx                3.6.1
numpy                   2.4.4
packaging               26.2
pandas                  3.0.2
parso                   0.8.6
pillow                  12.2.0
pip                     24.0
platformdirs            4.9.6
prompt_toolkit          3.0.52
propcache               0.4.1
psutil                  7.2.2
pure_eval               0.2.3
pyarrow                 24.0.0
Pygments                2.20.0
pyparsing               3.3.2
python-dateutil         2.9.0.post0
pytorch-forecasting     1.7.0
pytorch-lightning       2.6.1
PyYAML                  6.0.3
pyzmq                   27.1.0
scikit-base             0.13.2
scikit-learn            1.8.0
scipy                   1.17.1
setuptools              65.5.0
six                     1.17.0
stack-data              0.6.3
sympy                   1.14.0
threadpoolctl           3.6.0
torch                   2.11.0
torchmetrics            1.9.0
tornado                 6.5.5
tqdm                    4.67.3
traitlets               5.14.3
typing_extensions       4.15.0
tzdata                  2026.2
wcwidth                 0.6.0
yarl                    1.23.0
```

Agora, finalmente, iniciamos os códigos do projeto.   

## **1. Importações**

In [2]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

import lightning.pytorch as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

import matplotlib.pyplot as plt

C:\projects\Libellula\.venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## **2. Loading de dataset original**

In [3]:
# Carregar dataset base (OHLC + indicadores)
df = pd.read_csv("C:/projects/Libellula/data/processed/financial_tools_datset.csv")

# Ordenar temporalmente (CRÍTICO)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.sort_values("Date").reset_index(drop=True)

# Converter data
df["Date"] = pd.to_datetime(df["Date"])

# Criar índice temporal numérico (exigência do TFT)
df["time_idx"] = np.arange(len(df))

# Criar target → retorno futuro (melhor que preço)
df["target"] = df["Price"].pct_change().shift(-1)

# Remover NaNs
df = df.dropna().reset_index(drop=True)
    
# ID da série (necessário para TFT)
df["series"] = "asset_1"

df.head()

,Date,Price,Open,High,Low,Change %,short_mavg,long_mavg,signal,EMA10,...,RSI200,%K10,%D10,%K30,%D30,%K200,%D200,time_idx,target,series
0,2020-11-18,1.1852,1.1863,1.1893,1.1849,-0.08%,1.18299,1.178573,1.0,1.182132,...,55.210834,67.142857,75.588103,78.233438,79.284963,88.299419,88.541667,201,0.001772,asset_1
1,2020-11-19,1.1873,1.1854,1.1883,1.1816,0.18%,1.18351,1.178660,1.0,1.183072,...,55.316146,72.727273,73.647562,84.858044,81.388013,89.825581,89.026163,202,-0.001684,asset_1
2,2020-11-20,1.1853,1.1875,1.1892,1.1850,-0.17%,1.18332,1.178577,1.0,1.183477,...,55.191932,61.363636,67.077922,78.548896,80.546793,88.372093,88.832364,203,-0.001097,asset_1
3,2020-11-23,1.1840,1.1853,1.1907,1.1800,-0.11%,1.18359,1.178417,1.0,1.183572,...,55.111089,58.641975,64.244295,74.447950,79.284963,87.427326,88.541667,204,0.004054,asset_1
4,2020-11-24,1.1888,1.1842,1.1896,1.1837,0.41%,1.18433,1.178380,1.0,1.184523,...,55.353767,88.271605,69.425739,89.589905,80.862250,90.915698,88.905039,205,0.002103,asset_1


## **3. Configuração temporal**

In [4]:
max_encoder_length = 64
max_prediction_length = 1

## 4. **Criação do dataset do TFT**

```python
cols_obj = df.select_dtypes(include=["object"]).columns
```
**O código seguinte é de suma importância para o treinamento do TFT preditivo,** pois ele trabalha apenas com valores numéricos, então temos que nos livrar de eventuais strings dentro do dataset.
1) Selecionamos todas as colunas com tipo "object" (strings), que não são compatíveis com o modelo.
```python
for col in cols_obj:
    df[col] = (
        df[col]
        .astype(str)
```
2) Garante que todos os valores são tratados como string para permitir operações de limpeza.
```python
        .str.replace("%", "", regex=False)
```
3) Remove símbolos não numéricos (ex: %), que impedem conversão para float.
```python
        .str.replace(",", ".", regex=False)
```
4) Padroniza separadores decimais (vírgula → ponto), garantindo formato numérico válido.
```python
        .str.extract(r"([-+]?\d*\.?\d+)")[0]
```
5) Extrai apenas a parte numérica da string usando regex (remove qualquer outro caractere residual).
```python
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")
```
6) Converte os valores para numérico (float). Valores inválidos viram NaN.
```python
    df = df.dropna().reset_index(drop=True)
```
7) Remove linhas com NaN gerados na conversão e reorganiza o índice.

#### Resultado:
Dataset totalmente numérico, compatível com normalização e treinamento do modelo.

#### Abaixo, aplicamos o código acima explicado

In [5]:
cols_obj = df.select_dtypes(include=["object"]).columns

for col in cols_obj:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.extract(r"([-+]?\d*\.?\d+)")[0]  # <-- pega só número
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().reset_index(drop=True)
df["time_idx"] = np.arange(len(df))

C:\Users\Miguel\AppData\Local\Temp\ipykernel_36356\1439733057.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_obj = df.select_dtypes(include=["object"]).columns


In [6]:
df.select_dtypes(include=["object"])

""
0
1
2
3
4
...
1361
1362
1363
1364


In [7]:
# Definir colunas que o modelo usa
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

training = TimeSeriesDataSet(
    df.iloc[:train_end],
    time_idx="time_idx",
    target="target",
    group_ids=["series"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=["time_idx"],

    time_varying_unknown_reals=[
        col for col in df.columns
        if col not in ["Date", "series", "target"]
    ]
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df.iloc[:val_end],
    min_prediction_idx=train_end,
    stop_randomization=True,
)

test = TimeSeriesDataSet.from_dataset(
    training,
    df,
    min_prediction_idx=val_end,
    stop_randomization=True,
)

## **5. Dataloader**

In [8]:
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=64,
    num_workers=0
)

validation_dataloader = validation.to_dataloader(
    train=False,
    batch_size=64,
    num_workers=0
)

test_dataloader = test.to_dataloader(
    train=False,
    batch_size=64,
    num_workers=0
)

## 6. **Criar modelo TFT**

In [9]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


## 7. **Callbacks**

In [10]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min"
)

lr_logger = LearningRateMonitor()

## **8. Trainer**

In [11]:
trainer = pl.Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_logger]
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## 9. **Treinamento**

In [12]:
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=validation_dataloader
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 0      | train | 0    
3  | prescalers                         | ModuleDict                      | 800    | train | 0    
4  | static_variable_selection          | VariableSelectionNetwork        | 0      | train | 0    
5  | encoder_variable_selection         | VariableSelectionNetwork        | 58.5 K | train | 0 

Sanity Checking DataLoader 0:   0%|                                                              | 0/2 [00:00<?, ?it/s]

C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\loops\fit_loop.py:317: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████████████████████████████████| 13/13 [00:11<00:00,  1.15it/s, v_num=37, train_loss_step=0.00185]
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|███████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  5.02it/s]
Epoch 1: 100%|█| 13/13 [00:09<00:00,  1.42it/s, v_num=37, train_loss_step=0.00209, val_loss=0.00219, train_loss_epoch=0
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|███████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.74it/s]
Epoch 2: 100%|█| 13/13 [00:09<00:00,  1.

## 10. **Gerar previsões**

In [13]:
raw_predictions = tft.predict(
    validation_dataloader,
    mode="raw"
)

pred = raw_predictions["prediction"].numpy()
mean_pred = pred.mean(axis=2)

C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


## 11. **Criar dataset de previsões**

In [14]:
# Criar índice alinhado com saída do modelo
pred_index = df.index[train_end:val_end]

df_pred = pd.DataFrame(index=pred_index)

df_pred["prediction"] = mean_pred.flatten()

# retorno previsto (já é target, mas mantemos consistência)
df_pred["pred_return"] = df_pred["prediction"]

# confiança simples (magnitude)
df_pred["confidence"] = np.abs(df_pred["pred_return"])

df_pred.head()

,prediction,pred_return,confidence
956,0.000320,0.000320,0.000320
957,0.000287,0.000287,0.000287
958,0.000234,0.000234,0.000234
959,0.000179,0.000179,0.000179
960,0.000114,0.000114,0.000114


## 12. **Indexação temporal das previsões**

In [15]:
df_pred["timestamp"] = df["Date"].iloc[train_end:val_end].values
df_pred = df_pred.set_index("timestamp").sort_index()

assert df_pred.index.is_monotonic_increasing
assert not df_pred.index.has_duplicates
df_pred

,prediction,pred_return,confidence
timestamp,,,
2024-07-18,0.000320,0.000320,0.000320
2024-07-19,0.000287,0.000287,0.000287
2024-07-22,0.000234,0.000234,0.000234
2024-07-23,0.000179,0.000179,0.000179
2024-07-24,0.000114,0.000114,0.000114
...,...,...,...
2025-04-24,0.001216,0.001216,0.001216
2025-04-25,0.001187,0.001187,0.001187
2025-04-28,0.001359,0.001359,0.001359


### Verificações de integridade

As previsões devem possuir timestamps únicos e em ordem crescente.

In [16]:
print(df_pred.index.min(), df_pred.index.max())

2024-07-18 00:00:00 2025-04-30 00:00:00


In [17]:
print(len(df_pred))

205


## 13. **Verificar dataset preditivo**

In [18]:
df_pred.head()

,prediction,pred_return,confidence
timestamp,,,
2024-07-18,0.000320,0.000320,0.000320
2024-07-19,0.000287,0.000287,0.000287
2024-07-22,0.000234,0.000234,0.000234
2024-07-23,0.000179,0.000179,0.000179
2024-07-24,0.000114,0.000114,0.000114


O merge com o envelope de risco é responsabilidade do pipeline independente de estado para RL.

In [19]:
print(df_pred.index.min(), df_pred.index.max())

2024-07-18 00:00:00 2025-04-30 00:00:00


## 14. **Salvar dataset preditivo**

In [20]:
PREDICTIONS_PATH = Path("C:/projects/Libellula/data/processed/predictions/tft_predictions.parquet")

In [21]:
PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
df_pred.to_parquet(PREDICTIONS_PATH)
PREDICTIONS_PATH

WindowsPath('C:/projects/Libellula/data/processed/predictions/tft_predictions.parquet')

## 15. **Salvar modelo**

In [22]:
torch.save(tft.state_dict(), "tft_model.pth")

test_raw_predictions = tft.predict(test_dataloader, mode="raw")
test_quantile_predictions = test_raw_predictions["prediction"].numpy()[:, 0, :]
test_returns = df["target"].iloc[val_end:].to_numpy()
assert len(test_quantile_predictions) == len(test_returns)

quantiles = np.asarray(tft.loss.quantiles)
median_index = np.abs(quantiles - 0.50).argmin()
point_predictions = test_quantile_predictions[:, median_index]
correlation = np.corrcoef(point_predictions, test_returns)[0, 1] if np.std(point_predictions) and np.std(test_returns) else np.nan
predictive_diagnostics = pd.Series({
    "mae": np.abs(point_predictions - test_returns).mean(),
    "rmse": np.sqrt(np.mean((point_predictions - test_returns) ** 2)),
    "directional_accuracy": ((point_predictions > 0) == (test_returns > 0)).mean(),
    "prediction_return_correlation": correlation,
})
quantile_calibration = pd.DataFrame({
    "nominal_coverage": quantiles,
    "empirical_coverage": [(test_returns <= test_quantile_predictions[:, i]).mean() for i in range(len(quantiles))],
})
print(predictive_diagnostics)
print(quantile_calibration)

C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


mae                              0.003155
rmse                             0.004227
directional_accuracy             0.487805
prediction_return_correlation   -0.031956
dtype: float64
   nominal_coverage  empirical_coverage
0              0.02            0.024390
1              0.10            0.053659
2              0.25            0.243902
3              0.50            0.497561
4              0.75            0.770732
5              0.90            0.892683
6              0.98            0.975610
